## 📑 Core Legal Module
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AoAKHg)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
%python
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
%python
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col, regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()


from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace
import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim
import boto3
import os
# spark.conf.set("spark.sql.ansi.enabled", "false")

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

unionRefDate =  spark.sql(f'''
                              SELECT min(BusinessDate) Ref
                              FROM `{env_prefix}-cdh-delivery`.repoint.core_mastertable m
						WHERE BusinessDate <= {run_date!r}
						AND BusinessDate > last_day(add_months({run_date!r},-2))
                              ''').collect()[0]['Ref']
unionRefDate = unionRefDate.strftime('%Y-%m-%d')

###### Schema of the table

In [0]:
table_name = "Core_LegalModule"

schema = (
"BusinessDate string,"
"AgreementKey string,"
"AssignmentDate_dp date,"
"AssignmentDate_agog date,"
"AssignmentDate_ktsx date,"
"AssignmentDate_pros date,"
"CurrentAgency_dp string,"
"CurrentAgency_agog string,"
"CurrentAgency_ktsx string,"
"CurrentAgency_pros string,"
"CurrentStage_dp string,"
"CurrentStage_agog string,"
"CurrentStage_ktsx string,"
"CurrentStage_pros string,"
"DP_ProsAdun date,"
"DP_OrisAdun date,"
"DP_CancelAdun date,"
"KSX_ProsAdun date,"
"KSX_OrisAdun date,"
"KSX_CancelAdun date,"
"EkkrPairA_EntryDate_dp date,"
"EkkrPairA_ExitDate_dp date,"
"EkkrPairD_EntryDate_dp date,"
"EkkrPairD_ExitDate_dp date,"
"EkkrPairG_EntryDate_dp date,"
"EkkrPairG_ExitDate_dp date,"
"AdynTaktEkkr1_dp date,"
"AdynTaktEkkr2_dp date,"
"AdynTaktEkkr3_dp date,"
"AdynTaktEkkr4_dp date,"
"EkkrPairB_EntryDate_agog date,"
"EkkrPairB_ExitDate_agog date,"
"EkkrPairC_EntryDate_agog date,"
"EkkrPairC_ExitDate_agog date,"
"EkkrPairE_EntryDate_agog date,"
"EkkrPairE_ExitDate_agog date,"
"EkkrPairF_EntryDate_agog date,"
"EkkrPairF_ExitDate_agog date,"
"EkkrPairH_EntryDate_agog date,"
"EkkrPairH_ExitDate_agog date,"
"EkkrPairI_EntryDate_agog date,"
"EkkrPairI_ExitDate_agog date,"
"EkkrPairL_EntryDate_agog date,"
"EkkrPairL_ExitDate_agog date,"
"EkkrPairM_EntryDate_agog date,"
"EkkrPairM_ExitDate_agog date,"
"AdynTaktEkkr1_agog date,"
"AdynTaktEkkr2_agog date,"
"EkkrPairA_EntryDate_ktsx date,"
"EkkrPairA_ExitDate_ktsx date,"
"EkkrPairD_EntryDate_ktsx date,"
"EkkrPairD_ExitDate_ktsx date,"
"EkkrPairG_EntryDate_ktsx date,"
"EkkrPairG_ExitDate_ktsx date,"
"EkkrPairJ_EntryDate_ktsx date,"
"EkkrPairJ_ExitDate_ktsx date,"
"EkkrPairK_EntryDate_ktsx date,"
"EkkrPairK_ExitDate_ktsx date,"
"EkkrPairL_EntryDate_ktsx date,"
"EkkrPairL_ExitDate_ktsx date,"
"EkkrPairN_EntryDate_ktsx date,"
"EkkrPairN_ExitDate_ktsx date,"
"AdynTaktEkkr1_ktsx date,"
"AdynTaktEkkr2_ktsx date,"
"AdynTaktEkkr3_ktsx date,"
"AdynTaktEkkr4_ktsx date,"
"AdynTaktEkkr5_ktsx date,"
"AdynTaktEkkr6_ktsx date,"
"EkkrPairA_EntryDate_pros date,"
"EkkrPairA_ExitDate_pros date,"
"EkkrPairD_EntryDate_pros date,"
"EkkrPairD_ExitDate_pros date,"
"EkkrPairG_EntryDate_pros date,"
"EkkrPairG_ExitDate_pros date,"
"EkkrPairJ_EntryDate_pros date,"
"EkkrPairJ_ExitDate_pros date,"
"EkkrPairK_EntryDate_pros date,"
"EkkrPairK_ExitDate_pros date,"
"InsertTimeStamp timestamp"																		
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"

delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date
#### type: ignore
)


##### Optimizer function

In [0]:
%python
# Helper function to create or replace table
def create_or_replace_table(table_name, df, partition_col):
    df.write.format("delta").mode("overwrite").partitionBy(partition_col).saveAsTable(table_name)
def create_or_replace_table_no_partition(table_name, df):
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)
def repair_and_analyze_table(table_name):
    spark.sql(f"REPAIR TABLE {table_name} SYNC METADATA")
    spark.sql(f"ANALYZE TABLE {table_name} COMPUTE STATISTICS")    

##### Master Table

In [0]:
mastertable = spark.sql(
    f'''
SELECT IFNULL(t.PartyTaxRegistrationNumber,t.CDI) ParentPartyId
	 , t.* 
     , CASE WHEN t.MenuCode IS NOT NULL THEN 1 else 0 END CMSFlag 
	 , CASE WHEN t.spv = 'GEMINI CORE SECURITISATION DAC' THEN 1 else 0 END GeMINiAccFlag
FROM `{env_prefix}-cdh-delivery`.repoint.core_mastertable t
WHERE t.BusinessDate = {unionRefDate!r}
''')
mastertable.createOrReplaceTempView("mastertable")

##### legalmoduleactivities

In [0]:
legalactions = spark.sql(
    f'''
SELECT *
FROM `{env_prefix}-reportinputfiles`.repoint.core_legalactionsmapping 
''')
legalactions.createOrReplaceTempView("legalactions")

##### Adynamies (legal actions)

In [0]:
casecode = spark.sql(
f'''
select distinct nc.ExposureKey, c.casecodekey, c.portfolioname
from `{env_prefix}-cdh`.primary.casecode c
inner join  `{env_prefix}-cdh`.primary.exposure nc on c.casecodekey = nc.casecode and nc.portfolioname = c.portfolioname
where c.portfolioname = 'Core Retail'
''')
casecode.createOrReplaceTempView("casecode")

In [0]:
HistoricalModificationBefore202309 = spark.sql(
f'''
SELECT DISTINCT h.OldExposureKey, h.NewExposureKey
FROM `{env_prefix}-cdh`.reference.corestatichistoricalmodificationbeforemigration h
WHERE h.OldExposureKey <> h.NewExposureKey AND h.OldExposureKey <> '' AND h.NewExposureKey <> ''
''')
HistoricalModificationBefore202309.createOrReplaceTempView("HistoricalModificationBefore202309")  

In [0]:
extlpfxb364_ever = spark.sql(
f'''
SELECT 
    OldExposureKey,
    NewExposureKey,
    CASE 
        -- If the value matches the format 'YYYY-MM-DD HH:mm:ss.0'
        WHEN SUBSTRING(RestructuringDate, 5, 1) = '-' THEN try_cast(RestructuringDate AS date)
        -- If the value matches the format 'DD/MM/YYYY'
        WHEN SUBSTRING(RestructuringDate, 3, 1) = '/' THEN TO_DATE(RestructuringDate, 'dd/MM/yyyy')
        -- For any other format, assuming it's already in the desired format
    END AS RestructuringDate,
    RestructuringType,
    AgreementKey,
    BusinessDate
FROM  `{env_prefix}-cdh`.externalfiles.extlpfxb364
''')
extlpfxb364_ever.createOrReplaceTempView("extlpfxb364_ever")

In [0]:
replacements = spark.sql('''
select distinct
    OldExposureKey,
    NewExposureKey
from extlpfxb364_ever
union all
select distinct
    OldExposureKey,
    NewExposureKey
from HistoricalModificationBefore202309
''')

replacements.createOrReplaceTempView("replacements")

In [0]:
from pyspark.sql.functions import col
# Recursive function to perform the UNION ALL operation
def recursive_union(name, depth):
    if depth == 1:
        return spark.sql(f"SELECT DISTINCT NewExposureKey, OldExposureKey FROM {name}")
    else:
        # Perform the recursive call and alias the resulting DataFrame
        recursive_df = recursive_union(name, depth - 1).alias("m")
       
        # Join the recursive DataFrame with the original DataFrame
        # You need to register the recursive DataFrame as a temporary view to use it in the SQL query
        recursive_df.createOrReplaceTempView("m")
       
        return spark.sql(f"""
            SELECT DISTINCT NewExposureKey, OldExposureKey FROM {name}
            UNION ALL
            SELECT m.NewExposureKey, a.OldExposureKey
            FROM {name} a
           JOIN m ON m.OldExposureKey = a.NewExposureKey AND m.NewExposureKey <> a.OldExposureKey
 
        """)
depth = 20
replacements_full = recursive_union("replacements", depth)
# Remove duplicates and persist the result as a new DataFrame
replacements_full_distinct = replacements_full.distinct().orderBy('NewExposureKey', 'OldExposureKey')
replacements_full_distinct.createOrReplaceTempView("replacements_full_distinct")

In [0]:
extcompany  = spark.sql(
f'''
select distinct EmployeePartyKey as CreatedBy, EmployeeFullName as ExternalCompany    
from `{env_prefix}-cdh`.primary.firmstructure fs
WHERE 1=1
AND EmployeeCategory = "External Company"
and SourceSystem = 'qca'
and fs.BusinessDate = {run_date!r}
'''
)  
extcompany.createOrReplaceTempView("extcompany")

In [0]:
interaction = spark.sql(
f'''
select ExposureKey
     , PortfolioName
     , InteractionDate AS ActivityDate
     , InteractionTypeKey AS ActivityTypeKey
     , TRIM(InteractionType) AS ActivityName
     , CDI
     , IsActiveFlag
     , ExternalCompanyKey 
     , COM.ExternalCompany 
     , c.LegalActivityMapping
     , c.LegalChapter
     , c.LegalActivityCategory
from `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) la
LEFT JOIN extcompany COM ON com.CreatedBy = CAST(la.ExternalCompanyKey AS STRING)
INNER JOIN LegalActions c on la.InteractionTypeKey = c.LegalActivityTypeKey 
where la.PortfolioName = 'Core Retail'
and la.InteractionDate <= {run_date!r} AND la.InteractionDate > '2018-10-31' 

UNION

select d.ExposureKey AS ExposureKey
     , la.PortfolioName
     , InteractionDate AS ActivityDate
     , InteractionTypeKey AS ActivityTypeKey
     , TRIM(InteractionType) AS ActivityName
     , CDI
     , IsActiveFlag
     , ExternalCompanyKey 
     , COM.ExternalCompany
     , c.LegalActivityMapping
     , c.LegalChapter
     , c.LegalActivityCategory
from `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) la
LEFT JOIN extcompany COM ON com.CreatedBy = CAST(la.ExternalCompanyKey AS STRING)
INNER JOIN LegalActions c on la.InteractionTypeKey = c.LegalActivityTypeKey 
INNER JOIN casecode d on d.casecodekey = la.casecode
where la.PortfolioName = 'Core Retail'
and la.InteractionDate <= {run_date!r} AND la.InteractionDate > '2018-10-31' 
''')
interaction.createOrReplaceTempView("interaction")

In [0]:
Activities = spark.sql(
f'''
SELECT acc.CDI, acc.PartyTaxRegistrationNumber, acc.ParentPartyId, acc.AgreementKey, act.ExposureKey, acc.AssetSegment, acc.CMSFlag, acc.AccountStatus ArrearsStatusENG, acc.Balance, acc.OnBalanceAmount, acc.OffBalanceAmount, acc.GeminiAccFlag, ActivityDate, ActivityTypeKey, ActivityName, LegalActivityMapping, LegalChapter, LegalActivityCategory, ExternalCompany, ReplacementsInheritanceFlag
FROM (
SELECT ExposureKey,ActivityDate, ActivityTypeKey, ActivityName, ExternalCompany, LegalActivityMapping, LegalChapter, LegalActivityCategory, 0 as ReplacementsInheritanceFlag		
FROM interaction a

UNION ALL

SELECT lpfx.NewExposureKey ExposureKey,ActivityDate, ActivityTypeKey, ActivityName, ExternalCompany, LegalActivityMapping, LegalChapter, LegalActivityCategory, 1 as ReplacementsInheritanceFlag	
FROM interaction a
JOIN replacements_full_distinct lpfx on a.ExposureKey = lpfx.OldExposureKey 

UNION ALL

SELECT AccountNumber ExposureKey,ActivityDate,trim(ActivityTypeKey) ActivityTypeKey,	ActivityName, ExternalCompany, c.LegalActivityMapping, c.LegalChapter, c.LegalActivityCategory, 0 as ReplacementsInheritanceFlag		
FROM `{env_prefix}-reportinputfiles`.repoint.core_otherlegalactivitiesprior2019 a
JOIN LegalActions c on trim(a.ActivityTypeKey) = c.LegalActivityTypeKey 

UNION ALL

SELECT lpfx.NewExposureKey ExposureKey,ActivityDate,trim(ActivityTypeKey) ActivityTypeKey,	ActivityName, ExternalCompany, c.LegalActivityMapping, c.LegalChapter, c.LegalActivityCategory, 1 as ReplacementsInheritanceFlag	
FROM `{env_prefix}-reportinputfiles`.repoint.core_otherlegalactivitiesprior2019 a
JOIN replacements_full_distinct lpfx on a.AccountNumber = lpfx.OldExposureKey
JOIN LegalActions c on trim(a.ActivityTypeKey) = c.LegalActivityTypeKey) act
INNER JOIN mastertable acc on act.ExposureKey = acc.ExposureKey
''')
Activities.createOrReplaceTempView("Activities")  

create_or_replace_table_no_partition(f"`{env_prefix}-cdh-delivery`.repoint.int_corelegalaction", Activities)
repair_and_analyze_table(f"`{env_prefix}-cdh-delivery`.repoint.int_corelegalaction")

In [0]:
Adynamies = spark.sql(
f'''
		SELECT * , rank() OVER(PARTITION BY ExposureKey, LegalChapter ORDER BY ActivityDate DESC, ExternalCompany ASC, ActivityTypeKey DESC) Rank --ORDER BY legal activity after date
        FROM (		
            SELECT      *
            FROM `{env_prefix}-cdh-delivery`.repoint.int_corelegalaction
            WHERE LegalActivityCategory = 'Αδυναμίες' AND ReplacementsInheritanceFlag = 1		
        ) a    
'''
)
Adynamies.createOrReplaceTempView("Adynamies")

In [0]:
Adynamies_anag = spark.sql(
f'''
		SELECT * 
		FROM (
			SELECT DISTINCT CDI, PartyTaxRegistrationNumber, AgreementKey, ActivityDate, ActivityTypeKey, ActivityName, LegalActivityMapping, LegalChapter, ExternalCompany
			,rank() OVER(PARTITION BY AgreementKey, LegalChapter ORDER BY ActivityDate DESC, ExternalCompany ASC, ActivityTypeKey DESC) Rank
			FROM Adynamies
			WHERE Rank = 1
		) n
'''
)
Adynamies_anag.createOrReplaceTempView("Adynamies_anag")

In [0]:
Adynamies_la = spark.sql(
f'''
		SELECT
			 AgreementKey
			,`ΔΠ_ΑΚΥΡΩΣΗ ΑΔΥΝΑΜΙΑΣ ΕΠΙΛΥΣΗΣ` DP_CancelAdun
			,`ΔΠ_ΟΡΙΣΤΙΚΗ ΑΔΥΝΑΜΙΑ ΕΠΙΛΥΣΗΣ` DP_OrisAdun
			,`ΔΠ_ΠΡΟΣΩΡΙΝΗ ΑΔΥΝΑΜΙΑ ΕΠΙΛΥΣΗΣ` DP_ProsAdun
			,`ΚΣΧ_ΑΚΥΡΩΣΗ ΑΔΥΝΑΜΙΑΣ ΕΠΙΛΥΣΗΣ` KSX_CancelAdun
			,`ΚΣΧ_ΟΡΙΣΤΙΚΗ ΑΔΥΝΑΜΙΑ ΕΠΙΛΥΣΗΣ` KSX_OrisAdun
			,`ΚΣΧ_ΠΡΟΣΩΡΙΝΗ ΑΔΥΝΑΜΙΑ ΕΠΙΛΥΣΗΣ` KSX_ProsAdun
		FROM (
			SELECT  AgreementKey
			,LegalChapter
			,ActivityDate
			FROM Adynamies_anag
			WHERE Rank = 1
		) as TableToPivot
		pivot(
			MIN(ActivityDate) FOR LegalChapter in(
				'ΔΠ_ΑΚΥΡΩΣΗ ΑΔΥΝΑΜΙΑΣ ΕΠΙΛΥΣΗΣ',
				'ΔΠ_ΟΡΙΣΤΙΚΗ ΑΔΥΝΑΜΙΑ ΕΠΙΛΥΣΗΣ',
				'ΔΠ_ΠΡΟΣΩΡΙΝΗ ΑΔΥΝΑΜΙΑ ΕΠΙΛΥΣΗΣ',
				'ΚΣΧ_ΑΚΥΡΩΣΗ ΑΔΥΝΑΜΙΑΣ ΕΠΙΛΥΣΗΣ',
				'ΚΣΧ_ΟΡΙΣΤΙΚΗ ΑΔΥΝΑΜΙΑ ΕΠΙΛΥΣΗΣ',
				'ΚΣΧ_ΠΡΟΣΩΡΙΝΗ ΑΔΥΝΑΜΙΑ ΕΠΙΛΥΣΗΣ'
			)
		) 
'''
)
Adynamies_la.createOrReplaceTempView("Adynamies_la")

##### Legal Process

In [0]:
Max_BD_pre_Account = spark.sql(
f'''
        SELECT step.ExposureKey, MAX(step.BusinessDate) MaxBD
        FROM `{env_prefix}-cdh`.primary.legalprocessstep step
        INNER JOIN `{env_prefix}-cdh`.primary.legalprocessstage stage ON step.BusinessDate=stage.BusinessDate AND step.LegalStageKey=stage.LegalStageKey AND step.LegalProcessKey=stage.LegalProcessKey
		WHERE 1 = 1
            AND step.BusinessDate <= {run_date!r}
            AND step.PortfolioName = 'Core Retail'
        GROUP BY step.ExposureKey
''' )
Max_BD_pre_Account.createOrReplaceTempView("Max_BD_pre_Account")

In [0]:
LegalProcess = spark.sql(
f'''
		SELECT * 
		FROM (
            SELECT  DISTINCT step.sourcesystem, acc.AgreementKey
                ,   CAST(CONCAT(LEFT(step.LegalStepStartDate, 4), '-', RIGHT(LEFT(step.LegalStepStartDate, 6), 2), '-', RIGHT(step.LegalStepStartDate, 2)) AS DATE) LegalStepUserCreationDate
                ,   step.LegalStepDefinition AS LegalStep
                ,   stage.LegalStageDefinition AS LegalStepStage
                ,   stage.LegalStageStartDate AS LegalStageCreationDate
                ,   CAST(stage.LegalProcessStartDate AS DATE) AS LegalProcessCreationDate
                ,   stage.LegalProcessDefinitionCode AS LegalProcess
                ,   stage.ExternalCompanyAssignmentKey AS LegalProcessAgencyCode
                ,   fs.EmployeeFullName            AS LegalProcessAgency
                ,   LegalProcessStatus
                ,   MaxBD
           		,	rank() OVER(PARTITION BY acc.AgreementKey, step.LegalStepDefinition, stage.LegalProcessDefinitionCode ORDER BY CAST(stage.LegalProcessStartDate AS DATE) DESC) Rank2
             , 1 as rank1
            FROM `{env_prefix}-cdh`.primary.legalprocessstep step
            INNER JOIN `{env_prefix}-cdh`.primary.legalprocessstage stage ON step.BusinessDate=stage.BusinessDate AND step.LegalStageKey=stage.LegalStageKey AND step.LegalProcessKey=stage.LegalProcessKey
            INNER JOIN mastertable acc ON step.ExposureKey = acc.ExposureKey
            INNER JOIN `{env_prefix}-cdh`.primary.exposure e on step.ExposureKey = e.ExposureKey AND step.BusinessDate = e.BusinessDate            
            LEFT JOIN `{env_prefix}-cdh`.primary.firmstructure fs ON stage.ExternalCompanyAssignmentKey = fs.EmployeePartyKey AND fs.EmployeeCategory = 'External Company' AND step.SourceSystem = fs. SourceSystem AND step.BusinessDate = fs.BusinessDate 
            INNER JOIN Max_BD_pre_Account maxbd ON step.ExposureKey = maxbd.ExposureKey AND step.BusinessDate = maxbd.MaxBD AND stage.BusinessDate=maxbd.MaxBD
			WHERE 1 = 1
			AND stage.LegalProcessStatus = 'Active'
            AND step.PortfolioName = 'Core Retail'
            AND e.IsActiveFlag = true
            AND e.PartyRelationshipType = 'Primary Owner'
            AND step.LegalStepCreatedDate <= step.BusinessDate
            AND CAST(CONCAT(LEFT(LegalStepStartDate, 4), '-', RIGHT(LEFT(LegalStepStartDate, 6), 2), '-', RIGHT(LegalStepStartDate, 2)) AS DATE) <= {run_date!r}
		) a
		WHERE a.Rank1 = 1
'''
)
LegalProcess.createOrReplaceTempView("LegalProcess")

##### Entoles Anathesis

In [0]:
EntolesAnathesis_dp = spark.sql(
f'''
		SELECT DISTINCT 
			 AgreementKey
			,CASE WHEN LegalProcess = 'Διαταγή Πληρωμής' THEN LegalStepUserCreationDate END AS AssignmentDate_dp
			,CASE WHEN LegalProcess = 'Διαταγή Πληρωμής' THEN LegalProcessAgency END AS CurrentAgency_dp
			,CASE WHEN LegalProcess = 'Διαταγή Πληρωμής' THEN LegalStep END AS Entoli_dp
		FROM (
			SELECT * ,rank() OVER(PARTITION BY AgreementKey, LegalProcess ORDER BY LegalStepUserCreationDate DESC, LegalStep ASC, LegalProcessAgency ASC) Rank
		
            FROM LegalProcess
			WHERE LegalProcess = 'Διαταγή Πληρωμής'
			AND LegalStep IN(
			'Ολοκλήρωση Προετοιμασίας Ανάθεσης με εντολή για νέα Δ/Π_Ελλείψει απογράφου',
			'Ολοκλήρωση Προετοιμασίας Ανάθεσης',
			'Ολοκλήρωση Προετοιμασίας Ανάθεσης με εντολή για Εξώδικο',
			'Εντολή ΔΠ_Προκαθορισμένη Περίμετρος')
		) a
		WHERE a.Rank = 1 
'''
)
EntolesAnathesis_dp.createOrReplaceTempView("EntolesAnathesis_dp")

In [0]:
EntolesAnathesis_agog = spark.sql(
f'''
	SELECT DISTINCT 
			 AgreementKey
			,CASE WHEN LegalProcess = 'Τακτική Αγωγή απο την Τράπεζα' THEN LegalStepUserCreationDate END AS AssignmentDate_agog
			,CASE WHEN LegalProcess = 'Τακτική Αγωγή απο την Τράπεζα' THEN LegalProcessAgency END AS CurrentAgency_agog
			,CASE WHEN LegalProcess = 'Τακτική Αγωγή απο την Τράπεζα' THEN LegalStep END AS Entoli_agog
		FROM (
			SELECT * ,rank() OVER(PARTITION BY AgreementKey, LegalProcess ORDER BY LegalStepUserCreationDate DESC, LegalStep ASC, LegalProcessAgency ASC) Rank
            FROM LegalProcess
			WHERE LegalProcess = 'Τακτική Αγωγή απο την Τράπεζα'
			AND LegalStep IN(
			'Ολοκλήρωση προετοιμασίας ανάθεσης - Αγνώστου Διαμονής',
			'Ολοκλήρωση προετοιμασίας ανάθεσης - Ελλείψεως Συμβατικών')
		) a
		WHERE a.Rank = 1
'''
)
EntolesAnathesis_agog.createOrReplaceTempView("EntolesAnathesis_agog")

In [0]:
EntolesAnathesis_ktsx = spark.sql(
f'''
		SELECT DISTINCT 
			 AgreementKey
			,CASE WHEN LegalProcess = 'Επιβολή Κατάσχεσης' THEN LegalStepUserCreationDate END AS AssignmentDate_ktsx
			,CASE WHEN LegalProcess = 'Επιβολή Κατάσχεσης' THEN LegalProcessAgency END AS CurrentAgency_ktsx
			,CASE WHEN LegalProcess = 'Επιβολή Κατάσχεσης' THEN LegalStep END AS Entoli_ktsx
		FROM (
			SELECT * ,rank() OVER(PARTITION BY AgreementKey, LegalProcess ORDER BY LegalStepUserCreationDate DESC, LegalStep ASC, LegalProcessAgency ASC) Rank	
            FROM LegalProcess
			WHERE LegalProcess = 'Επιβολή Κατάσχεσης'
			AND LegalStep IN(
				'Ολοκλήρωση Προετοιμασίας Ανάθεσης',
				'Εντολή ΚΤΧΣ_Προκαθορισμένη Περίμετρος',
				'Εντολή επιβολής νέας κατάσχεσης'
			)
		) a
		WHERE a.Rank = 1
'''
)
EntolesAnathesis_ktsx.createOrReplaceTempView("EntolesAnathesis_ktsx")

In [0]:
EntolesAnathesis_pros = spark.sql(
f'''		
        SELECT DISTINCT
			 AgreementKey
			,CASE WHEN LegalProcess = 'Αναγκαστική Προσημείωση' THEN LegalStepUserCreationDate END AS AssignmentDate_pros
			,CASE WHEN LegalProcess = 'Αναγκαστική Προσημείωση' THEN LegalProcessAgency END AS CurrentAgency_pros 
			,CASE WHEN LegalProcess = 'Αναγκαστική Προσημείωση' THEN LegalStep END AS Entoli_pros
		FROM (
			SELECT * ,rank() OVER(PARTITION BY AgreementKey, LegalProcess ORDER BY LegalStepUserCreationDate DESC, LegalStep ASC, LegalProcessAgency ASC) Rank
            FROM LegalProcess
			WHERE LegalProcess = 'Αναγκαστική Προσημείωση'
			AND LegalStep IN('Ολοκλήρωση Προετοιμασίας Ανάθεσης')
		) a
		WHERE a.Rank = 1
'''
)
EntolesAnathesis_pros.createOrReplaceTempView("EntolesAnathesis_pros")

##### Current Stage

In [0]:
CurrentStage_dp = spark.sql(
f'''
		SELECT DISTINCT
			 AgreementKey
			,CASE WHEN LegalProcess = 'Διαταγή Πληρωμής' THEN LegalStepStage END AS CurrentStage_dp
		FROM (
			SELECT * ,rank() OVER(PARTITION BY AgreementKey, LegalProcess ORDER BY LegalStepUserCreationDate DESC, LegalStep ASC, LegalProcessAgency ASC) Rank		
            FROM LegalProcess
			WHERE LegalProcess = 'Διαταγή Πληρωμής'
		) a
		WHERE a.Rank = 1
'''
)
CurrentStage_dp.createOrReplaceTempView("CurrentStage_dp")

In [0]:
CurrentStage_agog = spark.sql(
f'''
		SELECT DISTINCT
			 AgreementKey
			,CASE WHEN LegalProcess = 'Τακτική Αγωγή απο την Τράπεζα' THEN LegalStepStage END AS CurrentStage_agog
		FROM (
			SELECT * ,rank() OVER(PARTITION BY AgreementKey, LegalProcess ORDER BY LegalStepUserCreationDate DESC, LegalStep ASC, LegalProcessAgency ASC) Rank
			FROM LegalProcess
			WHERE LegalProcess = 'Τακτική Αγωγή απο την Τράπεζα'
		) a
		WHERE a.Rank = 1
'''
)
CurrentStage_agog.createOrReplaceTempView("CurrentStage_agog")

In [0]:
CurrentStage_ktsx = spark.sql(
f'''
		SELECT DISTINCT
			 AgreementKey
			,CASE WHEN LegalProcess = 'Επιβολή Κατάσχεσης' THEN LegalStepStage END AS CurrentStage_ktsx
		FROM (
			SELECT * ,rank() OVER(PARTITION BY AgreementKey, LegalProcess ORDER BY LegalStepUserCreationDate DESC, LegalStep ASC, LegalProcessAgency ASC) Rank
			FROM LegalProcess
			WHERE LegalProcess = 'Επιβολή Κατάσχεσης'
		) a
		WHERE a.Rank = 1
'''
)
CurrentStage_ktsx.createOrReplaceTempView("CurrentStage_ktsx")

In [0]:
CurrentStage_pros = spark.sql(
f'''	
		SELECT DISTINCT 
			 AgreementKey
			,CASE WHEN LegalProcess = 'Αναγκαστική Προσημείωση' THEN LegalStepStage END AS CurrentStage_pros
		FROM (
			SELECT * ,rank() OVER(PARTITION BY AgreementKey, LegalProcess ORDER BY LegalStepUserCreationDate DESC, LegalStep ASC, LegalProcessAgency ASC) Rank
			FROM LegalProcess
			WHERE LegalProcess = 'Αναγκαστική Προσημείωση'
		) a
		WHERE a.Rank = 1
'''
)
CurrentStage_pros.createOrReplaceTempView("CurrentStage_pros")

##### Ekkremothtes

In [0]:
Ekkremotites_dp = spark.sql(
f'''
		SELECT DISTINCT
			 AgreementKey
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση` EkkrPairA_EntryDate_dp
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση` END EkkrPairA_ExitDate_dp
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο` EkkrPairD_EntryDate_dp
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο` END EkkrPairD_ExitDate_dp
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά` EkkrPairG_EntryDate_dp
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά` END EkkrPairG_ExitDate_dp
		FROM (
			SELECT 
				 AgreementKey
				,LegalStep
				,LegalStepUserCreationDate
				,AssignmentDate_dp
			FROM (
				SELECT lp.* ,e.AssignmentDate_dp ,rank() OVER(PARTITION BY lp.AgreementKey, lp.LegalStep ORDER BY lp.LegalStepUserCreationDate DESC, lp.LegalStep ASC) Rank
				FROM LegalProcess lp
				INNER JOIN EntolesAnathesis_dp e ON e.AgreementKey = lp.AgreementKey
				WHERE LegalProcess = 'Διαταγή Πληρωμής'
				AND LegalStep IN(
					'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση',
					'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση',
					'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο',
					'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο',
					'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά',
					'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά'
				)
				AND lp.LegalStepUserCreationDate >= e.AssignmentDate_dp
			) a
			WHERE a.Rank = 1
		) as TableToPivot
		pivot(
			MIN(LegalStepUserCreationDate) FOR LegalStep in(
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά')
		) --as pivotable
'''
)
Ekkremotites_dp.createOrReplaceTempView("Ekkremotites_dp")

In [0]:
Ekkremotites_agog = spark.sql(
f'''
		SELECT DISTINCT 
			AgreementKey
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση-Ελλ.Συμβατικών` EkkrPairB_EntryDate_agog
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση-Ελλ.Συμβατικών` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση-Ελλ.Συμβατικών` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση-Ελλ.Συμβατικών` END EkkrPairB_ExitDate_agog
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση-Αγνώστου Διαμονής` EkkrPairC_EntryDate_agog
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση-Αγνώστου Διαμονής` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση-Αγνώστου Διαμονής` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση-Αγνώστου Διαμονής` END EkkrPairC_ExitDate_agog
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο-Ελλ.Συμβατικών` EkkrPairE_EntryDate_agog
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο-Ελλ.Συμβατικών` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο-Ελλ.Συμβατικών` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο-Ελλ.Συμβατικών` END EkkrPairE_ExitDate_agog
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο-Αγνώστου Διαμονής` EkkrPairF_EntryDate_agog
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο-Αγνώστου Διαμονής` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο-Αγνώστου Διαμονής` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο-Αγνώστου Διαμονής` END EkkrPairF_ExitDate_agog
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά-Ελλ.Συμβατικών` EkkrPairH_EntryDate_agog
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά-Ελλ.Συμβατικών` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά-Ελλ.Συμβατικών` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά-Ελλ.Συμβατικών` END EkkrPairH_ExitDate_agog
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά-Αγνώστου Διαμονής` EkkrPairI_EntryDate_agog
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά-Αγνώστου Διαμονής` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά-Αγνώστου Διαμονής` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά-Αγνώστου Διαμονής` END EkkrPairI_ExitDate_agog
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων- Λοιπά` EkkrPairL_EntryDate_agog
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Λοιπά` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων- Λοιπά` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Λοιπά` END EkkrPairL_ExitDate_agog
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων- Λοιπά-Ελλ.Συμβατικών` EkkrPairM_EntryDate_agog
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Λοιπά-Ελλ.Συμβατικών` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων- Λοιπά-Ελλ.Συμβατικών` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Λοιπά-Ελλ.Συμβατικών` END EkkrPairM_ExitDate_agog
		FROM (
			SELECT 
				 AgreementKey
				,LegalStep
				,LegalStepUserCreationDate
				,AssignmentDate_agog
			FROM (
				SELECT 
					lp.* ,e.AssignmentDate_agog ,rank() OVER(PARTITION BY lp.AgreementKey, lp.LegalStep ORDER BY lp.LegalStepUserCreationDate DESC, lp.LegalStep ASC) Rank
				FROM LegalProcess lp
				INNER JOIN EntolesAnathesis_agog e ON e.AgreementKey = lp.AgreementKey
				WHERE LegalProcess = 'Τακτική Αγωγή απο την Τράπεζα'
				AND LegalStep IN(
					'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση-Ελλ.Συμβατικών',
					'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση-Ελλ.Συμβατικών',
					'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση-Αγνώστου Διαμονής',
					'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση-Αγνώστου Διαμονής',
					'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο-Ελλ.Συμβατικών',
					'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο-Ελλ.Συμβατικών',
					'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο-Αγνώστου Διαμονής',
					'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο-Αγνώστου Διαμονής',
					'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά-Ελλ.Συμβατικών',
					'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά-Ελλ.Συμβατικών',
					'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά-Αγνώστου Διαμονής',
					'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά-Αγνώστου Διαμονής',
					'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων- Λοιπά',
					'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Λοιπά',
					'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων- Λοιπά-Ελλ.Συμβατικών',
					'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Λοιπά-Ελλ.Συμβατικών'
				)
				AND lp.LegalStepUserCreationDate >= e.AssignmentDate_agog
			) a
			WHERE a.Rank = 1
		) as TableToPivot
		pivot(
			MIN(LegalStepUserCreationDate) FOR LegalStep in(
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση-Ελλ.Συμβατικών',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση-Ελλ.Συμβατικών',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση-Αγνώστου Διαμονής',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση-Αγνώστου Διαμονής',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο-Ελλ.Συμβατικών',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο-Ελλ.Συμβατικών',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο-Αγνώστου Διαμονής',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο-Αγνώστου Διαμονής',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά-Ελλ.Συμβατικών',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά-Ελλ.Συμβατικών',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά-Αγνώστου Διαμονής',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά-Αγνώστου Διαμονής',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων- Λοιπά',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Λοιπά',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων- Λοιπά-Ελλ.Συμβατικών',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Λοιπά-Ελλ.Συμβατικών'
			)
		) 
'''
)
Ekkremotites_agog.createOrReplaceTempView("Ekkremotites_agog")

In [0]:
Ekkremotites_ktsx = spark.sql(
f'''
    		SELECT DISTINCT 
			 AgreementKey
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση` EkkrPairA_EntryDate_ktsx
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση` END EkkrPairA_ExitDate_ktsx
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο` EkkrPairD_EntryDate_ktsx
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο` END EkkrPairD_ExitDate_ktsx
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά` `EkkrPairG_EntryDate_ktsx`
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά` END EkkrPairG_ExitDate_ktsx
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Πρωτότυπο Απόγραφο` EkkrPairJ_EntryDate_ktsx
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Πρωτότυπο Απόγραφο` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Πρωτότυπο Απόγραφο` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Πρωτότυπο Απόγραφο` END EkkrPairJ_ExitDate_ktsx
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Έγγραφα Εξασφαλίσεων` EkkrPairK_EntryDate_ktsx
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων -  Έγγραφα Εξασφαλίσεων` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Έγγραφα Εξασφαλίσεων` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων -  Έγγραφα Εξασφαλίσεων` END EkkrPairK_ExitDate_ktsx
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων- Λοιπά` EkkrPairL_EntryDate_ktsx
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Λοιπά` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων- Λοιπά` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Λοιπά` END EkkrPairL_ExitDate_ktsx
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Νομικά Έγγραφα` EkkrPairN_EntryDate_ktsx
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Νομικά Έγγραφα` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Νομικά Έγγραφα` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Νομικά Έγγραφα` END EkkrPairN_ExitDate_ktsx
		FROM (
			SELECT 
				 AgreementKey
				,LegalStep
				,LegalStepUserCreationDate
				,AssignmentDate_ktsx
			FROM (
				SELECT lp.* ,e.AssignmentDate_ktsx ,rank() OVER(PARTITION BY lp.AgreementKey, lp.LegalStep ORDER BY lp.LegalStepUserCreationDate DESC, lp.LegalStep ASC) Rank
				FROM LegalProcess lp
				INNER JOIN EntolesAnathesis_ktsx e ON e.AgreementKey = lp.AgreementKey
				WHERE LegalProcess = 'Επιβολή Κατάσχεσης'
				AND LegalStep IN(
					'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση',
					'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση',
					'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο',
					'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο',
					'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά',
					'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά',
					'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Πρωτότυπο Απόγραφο',
					'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Πρωτότυπο Απόγραφο',
					'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Έγγραφα Εξασφαλίσεων',
					'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων -  Έγγραφα Εξασφαλίσεων',
					'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων- Λοιπά',
					'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Λοιπά',
					'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Νομικά Έγγραφα',
					'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Νομικά Έγγραφα'
				)
				AND lp.LegalStepUserCreationDate >= e.AssignmentDate_ktsx
			) a
			WHERE a.Rank = 1
		) as TableToPivot
		pivot(
			MIN(LegalStepUserCreationDate) FOR LegalStep in(
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Πρωτότυπο Απόγραφο',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Πρωτότυπο Απόγραφο',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Έγγραφα Εξασφαλίσεων',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων -  Έγγραφα Εξασφαλίσεων',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων- Λοιπά',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Λοιπά',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Νομικά Έγγραφα',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Νομικά Έγγραφα'
			)
		)
'''
)
Ekkremotites_ktsx.createOrReplaceTempView("Ekkremotites_ktsx")

In [0]:
Ekkremotites_pros = spark.sql(
f'''
    		SELECT DISTINCT
			 AgreementKey
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση` EkkrPairA_EntryDate_pros
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση` END EkkrPairA_ExitDate_pros
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο` EkkrPairD_EntryDate_pros
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο` END EkkrPairD_ExitDate_pros
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά` EkkrPairG_EntryDate_pros
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά` END EkkrPairG_ExitDate_pros
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Πρωτότυπο Απόγραφο` EkkrPairJ_EntryDate_pros
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Πρωτότυπο Απόγραφο` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Πρωτότυπο Απόγραφο` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Πρωτότυπο Απόγραφο` END EkkrPairJ_ExitDate_pros
			,`Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Έγγραφα Εξασφαλίσεων` EkkrPairK_EntryDate_pros
			,CASE WHEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων -  Έγγραφα Εξασφαλίσεων` >= `Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Έγγραφα Εξασφαλίσεων` THEN `Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων -  Έγγραφα Εξασφαλίσεων` END EkkrPairK_ExitDate_pros
		FROM (
			SELECT AgreementKey
			,LegalStep
			,LegalStepUserCreationDate
			,AssignmentDate_pros
			FROM (
				SELECT lp.* ,e.AssignmentDate_pros ,rank() OVER(PARTITION BY lp.AgreementKey, lp.LegalStep ORDER BY lp.LegalStepUserCreationDate DESC, lp.LegalStep ASC) Rank
			FROM LegalProcess lp
			INNER JOIN EntolesAnathesis_pros e ON e.AgreementKey = lp.AgreementKey
			WHERE LegalProcess = 'Αναγκαστική Προσημείωση'
				AND LegalStep IN(
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Πρωτότυπο Απόγραφο',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Πρωτότυπο Απόγραφο',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Έγγραφα Εξασφαλίσεων',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων -  Έγγραφα Εξασφαλίσεων'
			)
			AND lp.LegalStepUserCreationDate >= e.AssignmentDate_pros) a
			WHERE a.Rank = 1
		) as TableToPivot
		pivot(
			MIN(LegalStepUserCreationDate) FOR LegalStep in(
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Κίνηση',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Κίνηση',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Εξώδικο',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Εξώδικο',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Συμβατικά',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Συμβατικά',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Πρωτότυπο Απόγραφο',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων - Πρωτότυπο Απόγραφο',
				'Ελλείψεις/ εκκρεμότητες Φυσικών Φακέλων - Έγγραφα Εξασφαλίσεων',
				'Τακτοποίηση εκκρεμοτήτων Φυσικών Φακέλων -  Έγγραφα Εξασφαλίσεων'
			)
		)
'''
)
Ekkremotites_pros.createOrReplaceTempView("Ekkremotites_pros")

##### Adynamies

In [0]:
Adynamies_dp = spark.sql(
f'''
		SELECT DISTINCT 
			 AgreementKey
			,`Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Κίνηση` AdynTaktEkkr1_dp
			,`Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Συμβατικά` AdynTaktEkkr2_dp
			,`Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Εξώδικο` AdynTaktEkkr3_dp
			,`Αδυναμία Εύρεσης Συμβατικών Εγγράφων` AdynTaktEkkr4_dp
		FROM (
			SELECT 
				 AgreementKey
				,LegalStep
				,LegalStepUserCreationDate
				,AssignmentDate_dp
			FROM (
				SELECT lp.* ,e.AssignmentDate_dp ,rank() OVER(PARTITION BY lp.AgreementKey, lp.LegalStep ORDER BY lp.LegalStepUserCreationDate DESC, lp.LegalStep ASC) Rank
				FROM LegalProcess lp
				INNER JOIN EntolesAnathesis_dp e ON e.AgreementKey = lp.AgreementKey
				WHERE LegalProcess = 'Διαταγή Πληρωμής'
				AND LegalStep IN(
					'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Κίνηση',
					'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Συμβατικά',
					'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Εξώδικο',
					'Αδυναμία Εύρεσης Συμβατικών Εγγράφων'
				)
				AND lp.LegalStepUserCreationDate>=e.AssignmentDate_dp
			) a
			WHERE a.Rank = 1
		) as TableToPivot
		pivot(
			MIN(LegalStepUserCreationDate) FOR LegalStep in(
				'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Κίνηση',
				'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Συμβατικά',
				'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Εξώδικο',
				'Αδυναμία Εύρεσης Συμβατικών Εγγράφων')
		)
'''
)
Adynamies_dp.createOrReplaceTempView("Adynamies_dp")

In [0]:
Adynamies_agog = spark.sql(
f'''

		SELECT DISTINCT 
			 AgreementKey
			,`Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Εξώδικο-Ελλ.Συμβατικών` AdynTaktEkkr1_agog
			,`Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Λοιπά` AdynTaktEkkr2_agog
		FROM (
			SELECT 
				 AgreementKey
				,LegalStep
				,LegalStepUserCreationDate
				,AssignmentDate_agog
			FROM (
				SELECT lp.* ,e.AssignmentDate_agog ,rank() OVER(PARTITION BY lp.AgreementKey, lp.LegalStep ORDER BY lp.LegalStepUserCreationDate DESC, lp.LegalStep ASC) Rank
				FROM LegalProcess lp
				INNER JOIN EntolesAnathesis_agog e ON e.AgreementKey = lp.AgreementKey
				WHERE LegalProcess = 'Τακτική Αγωγή απο την Τράπεζα'
				AND LegalStep IN(
					'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Εξώδικο-Ελλ.Συμβατικών',
					'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Λοιπά'
				)
				AND lp.LegalStepUserCreationDate >= e.AssignmentDate_agog
			) a
			WHERE a.Rank = 1
		) as TableToPivot
		pivot(
			MIN(LegalStepUserCreationDate) FOR LegalStep in(
				'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Εξώδικο-Ελλ.Συμβατικών',
				'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Λοιπά')
		) 
'''
)
Adynamies_agog.createOrReplaceTempView("Adynamies_agog")

In [0]:
Adynamies_ktsx = spark.sql(
f'''
		SELECT DISTINCT 
			 AgreementKey
			,`Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Κίνηση` AdynTaktEkkr1_ktsx
			,`Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Συμβατικά` AdynTaktEkkr2_ktsx
			,`Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Εξώδικο` AdynTaktEkkr3_ktsx
			,`Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Λοιπά` AdynTaktEkkr4_ktsx
			,`Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Πρωτότυπο Απόγραφο` AdynTaktEkkr5_ktsx
			,`Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Έγγραφα Εξασφαλίσεων` AdynTaktEkkr6_ktsx
		FROM (
			SELECT 
				 AgreementKey
				,LegalStep
				,LegalStepUserCreationDate
				,AssignmentDate_ktsx
			FROM (
				SELECT lp.* ,e.AssignmentDate_ktsx ,rank() OVER(PARTITION BY lp.AgreementKey, lp.LegalStep ORDER BY lp.LegalStepUserCreationDate DESC, lp.LegalStep ASC) Rank
				FROM LegalProcess lp
				INNER JOIN EntolesAnathesis_ktsx e ON e.AgreementKey = lp.AgreementKey
				WHERE LegalProcess = 'Επιβολή Κατάσχεσης'
				AND LegalStep IN(
					'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Κίνηση',
					'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Συμβατικά',
					'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Εξώδικο',
					'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Λοιπά',
					'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Πρωτότυπο Απόγραφο',
					'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Έγγραφα Εξασφαλίσεων'
				)
				AND lp.LegalStepUserCreationDate >= e.AssignmentDate_ktsx
			) a
			WHERE a.Rank = 1
		) as TableToPivot
		pivot(
			MIN(LegalStepUserCreationDate) FOR LegalStep in(
				'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Κίνηση',
				'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Συμβατικά',
				'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Εξώδικο',
				'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Λοιπά',
				'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Πρωτότυπο Απόγραφο',
				'Αδυναμία τακτοποίησης εκκρεμοτήτων Φ.Φ - Έγγραφα Εξασφαλίσεων'
			)
		) 
'''
)
Adynamies_ktsx.createOrReplaceTempView("Adynamies_ktsx")

##### LegalModule

In [0]:
LegalModule = spark.sql(
f'''
		SELECT DISTINCT
       		{run_date!r} AS BusinessDate
			,a.AgreementKey
			,ent_dp.AssignmentDate_dp
			,ent_agog.AssignmentDate_agog
			,ent_ktsx.AssignmentDate_ktsx
			,ent_pros.AssignmentDate_pros
			,ent_dp.CurrentAgency_dp
			,ent_agog.CurrentAgency_agog
			,ent_ktsx.CurrentAgency_ktsx
			,ent_pros.CurrentAgency_pros
			,cs_dp.CurrentStage_dp
			,cs_agog.CurrentStage_agog
			,cs_ktsx.CurrentStage_ktsx
			,cs_pros.CurrentStage_pros
			,adyn_la.DP_ProsAdun
			,adyn_la.DP_OrisAdun
			,adyn_la.DP_CancelAdun
			,adyn_la.KSX_ProsAdun
			,adyn_la.KSX_OrisAdun
			,adyn_la.KSX_CancelAdun
			,ekkr_dp.EkkrPairA_EntryDate_dp
			,ekkr_dp.EkkrPairA_ExitDate_dp
			,ekkr_dp.EkkrPairD_EntryDate_dp
			,ekkr_dp.EkkrPairD_ExitDate_dp
			,ekkr_dp.EkkrPairG_EntryDate_dp
			,ekkr_dp.EkkrPairG_ExitDate_dp
			,adyn_dp.AdynTaktEkkr1_dp
			,adyn_dp.AdynTaktEkkr2_dp
			,adyn_dp.AdynTaktEkkr3_dp
			,adyn_dp.AdynTaktEkkr4_dp
			,ekkr_agog.EkkrPairB_EntryDate_agog
			,ekkr_agog.EkkrPairB_ExitDate_agog
			,ekkr_agog.EkkrPairC_EntryDate_agog
			,ekkr_agog.EkkrPairC_ExitDate_agog
			,ekkr_agog.EkkrPairE_EntryDate_agog
			,ekkr_agog.EkkrPairE_ExitDate_agog
			,ekkr_agog.EkkrPairF_EntryDate_agog
			,ekkr_agog.EkkrPairF_ExitDate_agog
			,ekkr_agog.EkkrPairH_EntryDate_agog
			,ekkr_agog.EkkrPairH_ExitDate_agog
			,ekkr_agog.EkkrPairI_EntryDate_agog
			,ekkr_agog.EkkrPairI_ExitDate_agog
			,ekkr_agog.EkkrPairL_EntryDate_agog
			,ekkr_agog.EkkrPairL_ExitDate_agog
			,ekkr_agog.EkkrPairM_EntryDate_agog
			,ekkr_agog.EkkrPairM_ExitDate_agog
			,adyn_agog.AdynTaktEkkr1_agog
			,adyn_agog.AdynTaktEkkr2_agog
			,ekkr_ktsx.EkkrPairA_EntryDate_ktsx
			,ekkr_ktsx.EkkrPairA_ExitDate_ktsx
			,ekkr_ktsx.EkkrPairD_EntryDate_ktsx
			,ekkr_ktsx.EkkrPairD_ExitDate_ktsx
			,ekkr_ktsx.EkkrPairG_EntryDate_ktsx
			,ekkr_ktsx.EkkrPairG_ExitDate_ktsx
			,ekkr_ktsx.EkkrPairJ_EntryDate_ktsx
			,ekkr_ktsx.EkkrPairJ_ExitDate_ktsx
			,ekkr_ktsx.EkkrPairK_EntryDate_ktsx
			,ekkr_ktsx.EkkrPairK_ExitDate_ktsx
			,ekkr_ktsx.EkkrPairL_EntryDate_ktsx
			,ekkr_ktsx.EkkrPairL_ExitDate_ktsx
			,ekkr_ktsx.EkkrPairN_EntryDate_ktsx
			,ekkr_ktsx.EkkrPairN_ExitDate_ktsx
			,adyn_ktsx.AdynTaktEkkr1_ktsx
			,adyn_ktsx.AdynTaktEkkr2_ktsx
			,adyn_ktsx.AdynTaktEkkr3_ktsx
			,adyn_ktsx.AdynTaktEkkr4_ktsx
			,adyn_ktsx.AdynTaktEkkr5_ktsx
			,adyn_ktsx.AdynTaktEkkr6_ktsx
			,ekkr_pros.EkkrPairA_EntryDate_pros
			,ekkr_pros.EkkrPairA_ExitDate_pros
			,ekkr_pros.EkkrPairD_EntryDate_pros
			,ekkr_pros.EkkrPairD_ExitDate_pros
			,ekkr_pros.EkkrPairG_EntryDate_pros
			,ekkr_pros.EkkrPairG_ExitDate_pros
			,ekkr_pros.EkkrPairJ_EntryDate_pros
			,ekkr_pros.EkkrPairJ_ExitDate_pros
			,ekkr_pros.EkkrPairK_EntryDate_pros
			,ekkr_pros.EkkrPairK_ExitDate_pros
		FROM mastertable a
		LEFT JOIN EntolesAnathesis_dp ent_dp ON ent_dp.AgreementKey = a.AgreementKey
		LEFT JOIN EntolesAnathesis_agog ent_agog ON ent_agog.AgreementKey = a.AgreementKey
		LEFT JOIN EntolesAnathesis_ktsx ent_ktsx ON ent_ktsx.AgreementKey = a.AgreementKey
		LEFT JOIN EntolesAnathesis_pros ent_pros ON ent_pros.AgreementKey = a.AgreementKey
		LEFT JOIN Ekkremotites_dp ekkr_dp ON ekkr_dp.AgreementKey = a.AgreementKey 
		LEFT JOIN Ekkremotites_agog ekkr_agog ON ekkr_agog.AgreementKey = a.AgreementKey 
		LEFT JOIN Ekkremotites_pros ekkr_pros ON ekkr_pros.AgreementKey = a.AgreementKey 
		LEFT JOIN Ekkremotites_ktsx ekkr_ktsx ON ekkr_ktsx.AgreementKey = a.AgreementKey 
		LEFT JOIN Adynamies_dp adyn_dp ON adyn_dp.AgreementKey = a.AgreementKey  
		LEFT JOIN Adynamies_agog adyn_agog ON adyn_agog.AgreementKey = a.AgreementKey  
		LEFT JOIN Adynamies_ktsx adyn_ktsx ON adyn_ktsx.AgreementKey = a.AgreementKey  
		LEFT JOIN Adynamies_la adyn_la ON adyn_la.AgreementKey = a.AgreementKey
		LEFT JOIN CurrentStage_dp cs_dp ON cs_dp.AgreementKey = a.AgreementKey
		LEFT JOIN CurrentStage_agog cs_agog ON cs_agog.AgreementKey = a.AgreementKey
		LEFT JOIN CurrentStage_ktsx cs_ktsx ON cs_ktsx.AgreementKey = a.AgreementKey
		LEFT JOIN CurrentStage_pros cs_pros ON cs_pros.AgreementKey = a.AgreementKey
		WHERE ent_dp.AgreementKey IS NOT NULL
		OR ent_agog.AgreementKey IS NOT NULL
		OR ent_ktsx.AgreementKey IS NOT NULL
		OR ent_pros.AgreementKey IS NOT NULL
		OR ent_dp.AgreementKey IS NOT NULL
		OR ekkr_dp.AgreementKey IS NOT NULL
		OR ekkr_agog.AgreementKey IS NOT NULL
		OR ekkr_pros.AgreementKey IS NOT NULL
		OR ekkr_ktsx.AgreementKey IS NOT NULL
		OR adyn_dp.AgreementKey IS NOT NULL
		OR adyn_agog.AgreementKey IS NOT NULL
		OR adyn_ktsx.AgreementKey IS NOT NULL
		OR adyn_la.AgreementKey IS NOT NULL
'''
)
LegalModule.createOrReplaceTempView("LegalModule")

##### final_df

In [0]:
final_df = spark.sql(
f'''
select
  cast(BusinessDate as string) as BusinessDate
, cast(AgreementKey as string) as AgreementKey
, cast(AssignmentDate_dp as date) as AssignmentDate_dp
, cast(AssignmentDate_agog as date) as AssignmentDate_agog
, cast(AssignmentDate_ktsx as date) as AssignmentDate_ktsx
, cast(AssignmentDate_pros as date) as AssignmentDate_pros
, cast(CurrentAgency_dp as string) as CurrentAgency_dp
, cast(CurrentAgency_agog as string) as CurrentAgency_agog
, cast(CurrentAgency_ktsx as string) as CurrentAgency_ktsx
, cast(CurrentAgency_pros as string) as CurrentAgency_pros
, cast(CurrentStage_dp as string) as CurrentStage_dp
, cast(CurrentStage_agog as string) as CurrentStage_agog
, cast(CurrentStage_ktsx as string) as CurrentStage_ktsx
, cast(CurrentStage_pros as string) as CurrentStage_pros
, cast(DP_ProsAdun as date) as DP_ProsAdun
, cast(DP_OrisAdun as date) as DP_OrisAdun
, cast(DP_CancelAdun as date) as DP_CancelAdun
, cast(KSX_ProsAdun as date) as KSX_ProsAdun
, cast(KSX_OrisAdun as date) as KSX_OrisAdun
, cast(KSX_CancelAdun as date) as KSX_CancelAdun
, cast(EkkrPairA_EntryDate_dp as date) as EkkrPairA_EntryDate_dp
, cast(EkkrPairA_ExitDate_dp as date) as EkkrPairA_ExitDate_dp
, cast(EkkrPairD_EntryDate_dp as date) as EkkrPairD_EntryDate_dp
, cast(EkkrPairD_ExitDate_dp as date) as EkkrPairD_ExitDate_dp
, cast(EkkrPairG_EntryDate_dp as date) as EkkrPairG_EntryDate_dp
, cast(EkkrPairG_ExitDate_dp as date) as EkkrPairG_ExitDate_dp
, cast(AdynTaktEkkr1_dp as date) as AdynTaktEkkr1_dp
, cast(AdynTaktEkkr2_dp as date) as AdynTaktEkkr2_dp
, cast(AdynTaktEkkr3_dp as date) as AdynTaktEkkr3_dp
, cast(AdynTaktEkkr4_dp as date) as AdynTaktEkkr4_dp
, cast(EkkrPairB_EntryDate_agog as date) as EkkrPairB_EntryDate_agog
, cast(EkkrPairB_ExitDate_agog as date) as EkkrPairB_ExitDate_agog
, cast(EkkrPairC_EntryDate_agog as date) as EkkrPairC_EntryDate_agog
, cast(EkkrPairC_ExitDate_agog as date) as EkkrPairC_ExitDate_agog
, cast(EkkrPairE_EntryDate_agog as date) as EkkrPairE_EntryDate_agog
, cast(EkkrPairE_ExitDate_agog as date) as EkkrPairE_ExitDate_agog
, cast(EkkrPairF_EntryDate_agog as date) as EkkrPairF_EntryDate_agog
, cast(EkkrPairF_ExitDate_agog as date) as EkkrPairF_ExitDate_agog
, cast(EkkrPairH_EntryDate_agog as date) as EkkrPairH_EntryDate_agog
, cast(EkkrPairH_ExitDate_agog as date) as EkkrPairH_ExitDate_agog
, cast(EkkrPairI_EntryDate_agog as date) as EkkrPairI_EntryDate_agog
, cast(EkkrPairI_ExitDate_agog as date) as EkkrPairI_ExitDate_agog
, cast(EkkrPairL_EntryDate_agog as date) as EkkrPairL_EntryDate_agog
, cast(EkkrPairL_ExitDate_agog as date) as EkkrPairL_ExitDate_agog
, cast(EkkrPairM_EntryDate_agog as date) as EkkrPairM_EntryDate_agog
, cast(EkkrPairM_ExitDate_agog as date) as EkkrPairM_ExitDate_agog
, cast(AdynTaktEkkr1_agog as date) as AdynTaktEkkr1_agog
, cast(AdynTaktEkkr2_agog as date) as AdynTaktEkkr2_agog
, cast(EkkrPairA_EntryDate_ktsx as date) as EkkrPairA_EntryDate_ktsx
, cast(EkkrPairA_ExitDate_ktsx as date) as EkkrPairA_ExitDate_ktsx
, cast(EkkrPairD_EntryDate_ktsx as date) as EkkrPairD_EntryDate_ktsx
, cast(EkkrPairD_ExitDate_ktsx as date) as EkkrPairD_ExitDate_ktsx
, cast(EkkrPairG_EntryDate_ktsx as date) as EkkrPairG_EntryDate_ktsx
, cast(EkkrPairG_ExitDate_ktsx as date) as EkkrPairG_ExitDate_ktsx
, cast(EkkrPairJ_EntryDate_ktsx as date) as EkkrPairJ_EntryDate_ktsx
, cast(EkkrPairJ_ExitDate_ktsx as date) as EkkrPairJ_ExitDate_ktsx
, cast(EkkrPairK_EntryDate_ktsx as date) as EkkrPairK_EntryDate_ktsx
, cast(EkkrPairK_ExitDate_ktsx as date) as EkkrPairK_ExitDate_ktsx
, cast(EkkrPairL_EntryDate_ktsx as date) as EkkrPairL_EntryDate_ktsx
, cast(EkkrPairL_ExitDate_ktsx as date) as EkkrPairL_ExitDate_ktsx
, cast(EkkrPairN_EntryDate_ktsx as date) as EkkrPairN_EntryDate_ktsx
, cast(EkkrPairN_ExitDate_ktsx as date) as EkkrPairN_ExitDate_ktsx
, cast(AdynTaktEkkr1_ktsx as date) as AdynTaktEkkr1_ktsx
, cast(AdynTaktEkkr2_ktsx as date) as AdynTaktEkkr2_ktsx
, cast(AdynTaktEkkr3_ktsx as date) as AdynTaktEkkr3_ktsx
, cast(AdynTaktEkkr4_ktsx as date) as AdynTaktEkkr4_ktsx
, cast(AdynTaktEkkr5_ktsx as date) as AdynTaktEkkr5_ktsx
, cast(AdynTaktEkkr6_ktsx as date) as AdynTaktEkkr6_ktsx
, cast(EkkrPairA_EntryDate_pros as date) as EkkrPairA_EntryDate_pros
, cast(EkkrPairA_ExitDate_pros as date) as EkkrPairA_ExitDate_pros
, cast(EkkrPairD_EntryDate_pros as date) as EkkrPairD_EntryDate_pros
, cast(EkkrPairD_ExitDate_pros as date) as EkkrPairD_ExitDate_pros
, cast(EkkrPairG_EntryDate_pros as date) as EkkrPairG_EntryDate_pros
, cast(EkkrPairG_ExitDate_pros as date) as EkkrPairG_ExitDate_pros
, cast(EkkrPairJ_EntryDate_pros as date) as EkkrPairJ_EntryDate_pros
, cast(EkkrPairJ_ExitDate_pros as date) as EkkrPairJ_ExitDate_pros
, cast(EkkrPairK_EntryDate_pros as date) as EkkrPairK_EntryDate_pros
, cast(EkkrPairK_ExitDate_pros as date) as EkkrPairK_ExitDate_pros
, cast(getdate() as timestamp) as InsertTimeStamp
from LegalModule
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_LegalModule: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["AgreementKey"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
LegalModule = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
LegalModule.createOrReplaceTempView("LegalModule")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/LegalModule/"
file_name = "LegalModule_" +run_date+ ".csv"
LegalModule.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)